# Speaker Verification using ResNet50

Identical to the ResNet34 baseline except the backbone is upgraded to **ResNet50**, which uses bottleneck residual blocks for even richer feature extraction.

---

## 1. Dataset Setup

In [ ]:
import os
import pandas as pd
import numpy as np

DATASET_ROOT = "/kaggle/input/datasets/tahmidulislamomi/ml-project-openslr-dataset"
BASE_AUDIO_DIR = os.path.join(DATASET_ROOT, "train-clean-100", "train-clean-100")
CSV_PATH = os.path.join(DATASET_ROOT, "train_pairs.csv")

print("BASE_AUDIO_DIR:", BASE_AUDIO_DIR)
print("CSV_PATH:", CSV_PATH)

## 2. Load and Inspect Training Data

Load the training CSV and confirm label balance.

In [ ]:
df = pd.read_csv(CSV_PATH)
print("Total rows:", len(df))
print("Label distribution:")
print(df["label"].value_counts())

## 3. Resolve Absolute Audio Paths

Strip the `train-clean-100/` prefix and join with the true dataset root.

In [ ]:
def to_absolute_path(rel_path):
    cleaned = rel_path.replace("train-clean-100/", "", 1)
    return os.path.join(BASE_AUDIO_DIR, cleaned)

df["path1_abs"] = df["audio_path_1"].apply(to_absolute_path)
df["path2_abs"] = df["audio_path_2"].apply(to_absolute_path)
print("Sample path:", df["path1_abs"].iloc[0])

## 4. Audio Transforms & Standardization

Random crop or zero-pad to exactly 3 seconds, then extract Log-Mel Spectrograms.

In [ ]:
import torch
import torchaudio.transforms as T

TARGET_SR = 16000
TARGET_LENGTH = TARGET_SR * 3  # 48000 samples

def crop_or_pad(audio):
    length = len(audio)
    if length > TARGET_LENGTH:
        start = np.random.randint(0, length - TARGET_LENGTH)
        audio = audio[start:start + TARGET_LENGTH]
    elif length < TARGET_LENGTH:
        audio = np.pad(audio, (0, TARGET_LENGTH - length), mode='constant')
    return audio

mel_transform = T.MelSpectrogram(
    sample_rate=16000, n_fft=400, hop_length=160, n_mels=80
)
amplitude_to_db = T.AmplitudeToDB()

## 5. Custom Dataset Class

In [ ]:
import soundfile as sf
from torch.utils.data import Dataset

class SpeakerPairDataset(Dataset):
    def __init__(self, dataframe, mel_transform, amplitude_to_db):
        self.df = dataframe
        self.mel_transform = mel_transform
        self.amplitude_to_db = amplitude_to_db

    def __len__(self):
        return len(self.df)

    def load_audio(self, path):
        audio, sr = sf.read(path)
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        audio = crop_or_pad(audio)
        audio = torch.tensor(audio).float().unsqueeze(0)
        mel = self.mel_transform(audio)
        return self.amplitude_to_db(mel)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        mel1 = self.load_audio(row["path1_abs"])
        mel2 = self.load_audio(row["path2_abs"])
        label = torch.tensor(row["label"]).float()
        return mel1, mel2, label

## 6. Initialize Training Dataset & DataLoader

In [ ]:
from torch.utils.data import DataLoader

dataset = SpeakerPairDataset(df, mel_transform, amplitude_to_db)
print("Dataset size:", len(dataset))

dataloader = DataLoader(
    dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True
)
print("DataLoader ready")

## 7. Model Architecture — ResNetSpeaker (ResNet50)

Only change from ResNet34: `resnet34()` → `resnet50()`.
ResNet50 uses **bottleneck blocks** (1×1 → 3×3 → 1×1 convolutions) instead of plain BasicBlocks, giving it 50 layers and a larger feature space.

> **Note:** ResNet50's final pooled features are 2048-dimensional (vs 512 for ResNet18/34), so the embedding layer automatically handles this via `in_features`.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class ResNetSpeaker(nn.Module):
    def __init__(self, embedding_dim=256):
        super().__init__()

        # *** ResNet50 instead of ResNet34 ***
        self.backbone = models.resnet50(weights=None)

        # Modify first conv to accept 1-channel Log-Mel input
        self.backbone.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        # ResNet50 fc input = 2048 (automatically detected)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        self.embedding = nn.Linear(in_features, embedding_dim)

    def forward(self, x):
        features = self.backbone(x)
        embedding = self.embedding(features)
        return F.normalize(embedding, p=2, dim=1)

## 8. Initialize Model on Device

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNetSpeaker(embedding_dim=256).to(device)
print("Using device:", device)
print(model)

## 9. Contrastive Loss Function

In [ ]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, emb1, emb2, label):
        distance = F.pairwise_distance(emb1, emb2)
        pos_loss = label * torch.pow(distance, 2)
        neg_loss = (1 - label) * torch.pow(torch.clamp(self.margin - distance, min=0.0), 2)
        return torch.mean(pos_loss + neg_loss)

## 10. Training Configuration

In [ ]:
model = ResNetSpeaker(256).to(device)
criterion = ContrastiveLoss(margin=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## 11. Training Loop

Train for **30 epochs** using contrastive loss. Live loss shown via `tqdm`.

In [ ]:
from tqdm import tqdm
import torch.nn.functional as F

num_epochs = 30
print_every = 50

loss_history = []   # average loss per epoch
acc_history  = []   # average training accuracy per epoch

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    progress_bar = tqdm(enumerate(dataloader),
                        total=len(dataloader),
                        desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch_idx, (mel1, mel2, labels) in progress_bar:
        mel1   = mel1.to(device)
        mel2   = mel2.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        emb1 = model(mel1)
        emb2 = model(mel2)
        loss = criterion(emb1, emb2, labels)
        loss.backward()
        optimizer.step()

        # Batch accuracy (cosine similarity threshold 0.5)
        with torch.no_grad():
            cosine_sim   = F.cosine_similarity(emb1, emb2)
            preds        = (cosine_sim > 0.5).float()
            total_correct  += (preds == labels).sum().item()
            total_samples  += labels.size(0)

        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

        if batch_idx % print_every == 0:
            print(f"Epoch {'{epoch+1}'} | Batch {'{batch_idx}'}/{'{len(dataloader)}'} | Loss: {'{loss.item():.4f}'}")

    avg_loss = total_loss / len(dataloader)
    avg_acc  = total_correct / total_samples

    loss_history.append(avg_loss)
    acc_history.append(avg_acc)

    print(f"\nEpoch [{'{epoch+1}'}/{'{num_epochs}'}] - Avg Loss: {'{avg_loss:.4f}'} | Avg Acc: {'{avg_acc:.4f}'}\n")

    # Auto-save after every epoch (handles Kaggle 12-hour timeout)
    torch.save({
        "model_state":     model.state_dict(),
        "epoch":           epoch,
        "optimizer_state": optimizer.state_dict(),
        "loss_history":    loss_history,
        "acc_history":     acc_history,
    }, "checkpoint_resnet50.pth")
    print(f"Checkpoint saved → epoch {epoch+1}")

## 13. Training Graphs

Two plots side-by-side:
- **Left** — Contrastive Loss per epoch (should trend downward)
- **Right** — Training accuracy per epoch (should trend upward)

> The accuracy is computed on training batches using cosine similarity threshold 0.5, so it reflects how well the model is learning to separate same/different speakers **during** training.

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(loss_history) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("ResNet50 — Training Metrics", fontsize=14, fontweight='bold')

# ── Plot 1: Loss per Epoch ────────────────────────────────────────────
axes[0].plot(epochs_range, loss_history, marker='o', color='tomato', linewidth=2)
axes[0].set_title("Contrastive Loss per Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

# ── Plot 2: Training Accuracy per Epoch ──────────────────────────────
axes[1].plot(epochs_range, acc_history, marker='o', color='steelblue', linewidth=2)
axes[1].set_title("Training Accuracy per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_metrics.png", dpi=150)
plt.show()
print("Graph saved to training_metrics.png")

## 12. Save the Trained Model

Save weights, epoch, and optimizer state to `checkpoint_resnet50.pth`.

In [ ]:
torch.save({
    "model_state": model.state_dict(),
    "epoch": epoch,
    "optimizer_state": optimizer.state_dict()
}, "checkpoint_resnet50.pth")

print("Model saved to checkpoint_resnet50.pth")

## 14. Embedding Distance Distribution

Plots a histogram of Euclidean distances between embedding pairs:
- 🔵 **Same speaker** (label=1) → distances should cluster near **0**
- 🔴 **Different speaker** (label=0) → distances should be **larger**

> A clear separation between the two histograms means the model has learned to distinguish speakers well.

In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F

model.eval()
same_dists, diff_dists = [], []

with torch.no_grad():
    for mel1, mel2, labels in dataloader:
        emb1 = model(mel1.to(device))
        emb2 = model(mel2.to(device))
        dists = F.pairwise_distance(emb1, emb2).cpu().tolist()
        for d, l in zip(dists, labels.tolist()):
            if l == 1:
                same_dists.append(d)
            else:
                diff_dists.append(d)
        if len(same_dists) > 5000:  # sample 5000 pairs — no need for full dataset
            break

plt.figure(figsize=(8, 5))
plt.hist(same_dists, bins=60, alpha=0.6, color='steelblue', label='Same Speaker (label=1)')
plt.hist(diff_dists, bins=60, alpha=0.6, color='tomato',    label='Different Speaker (label=0)')
plt.xlabel('Euclidean Distance')
plt.ylabel('Count')
plt.title('Embedding Distance Distribution After Training')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('distance_distribution.png', dpi=150)
plt.show()
print('Graph saved to distance_distribution.png')